# EP1 - Limpieza inicial (Colab / Jupyter)

Este notebook ejecuta el proceso de limpieza inicial y genera: `data/processed/movies_processed.csv`, `data/processed/tv_processed.csv` y `documentation/markdown/EP1_data_cleaning.md`.

Instrucciones rápidas:
- En Colab: sube los dos CSV (`netflix_movies_detailed_up_to_2025.csv` y `netflix_tv_shows_detailed_up_to_2025.csv`) usando el botón
 o el siguiente comando de subida.
- En Jupyter local: asegúrate de ejecutar el notebook desde la raíz del repo (donde está la carpeta `data/`).

In [3]:
import os

movies_path = None
tv_path = None

try:
    from google.colab import files
    # Running in Colab
    print('Entorno Colab detectado.')

    # Check for files already present in /content/
    colab_movie_filename = 'netflix_movies_detailed_up_to_2025.csv'
    colab_tv_filename = 'netflix_tv_shows_detailed_up_to_2025.csv'

    colab_movie_path = os.path.join('/content', colab_movie_filename)
    colab_tv_path = os.path.join('/content', colab_tv_filename)

    if os.path.exists(colab_movie_path):
        movies_path = colab_movie_path
        print(f'Ruta de películas (existente): {movies_path}')
    else:
        print(f'Advertencia: Archivo de películas no encontrado en {colab_movie_path}')

    if os.path.exists(colab_tv_path):
        tv_path = colab_tv_path
        print(f'Ruta de series de TV (existente): {tv_path}')
    else:
        print(f'Advertencia: Archivo de series de TV no encontrado en {colab_tv_path}')

    # If paths are still None or files were not found directly, prompt user to upload
    if movies_path is None or tv_path is None:
        print('Algunos archivos no se encontraron en /content/ o no se han subido. Por favor, súbelos.')
        # This will block execution until user uploads files
        uploaded = files.upload()
        for k in uploaded.keys():
            name = k.lower()
            if 'movie' in name and movies_path is None:
                movies_path = k
            if ('tv' in name or 'show' in name) and tv_path is None:
                tv_path = k

except ImportError:
    # Not in Colab, likely a local environment
    print('No es entorno Colab. Usando rutas locales.')
    # Original local paths (Windows specific)
    movies_path = r'C:\Users\shein\OneDrive\Documentos\GitHub\visualizacion-de-datos-StreamView-Analytics\data\netflix_movies_detailed_up_to_2025.csv'
    tv_path = r'C:\Users\shein\OneDrive\Documentos\GitHub\visualizacion-de-datos-StreamView-Analytics\data\netflix_tv_shows_detailed_up_to_2025.csv'

    # Added check for relative paths if the above absolute paths don't exist
    current_dir = os.getcwd()
    relative_data_dir = os.path.join(current_dir, 'data')
    relative_movies_path = os.path.join(relative_data_dir, 'netflix_movies_detailed_up_to_2025.csv')
    relative_tv_path = os.path.join(relative_data_dir, 'netflix_tv_shows_detailed_up_to_2025.csv')

    if not os.path.exists(movies_path) and os.path.exists(relative_movies_path):
        movies_path = relative_movies_path
        print(f'Usando ruta relativa de películas: {movies_path}')
    if not os.path.exists(tv_path) and os.path.exists(relative_tv_path):
        tv_path = relative_tv_path
        print(f'Usando ruta relativa de series de TV: {tv_path}')

    if not os.path.exists(movies_path):
        print('Advertencia: Archivo de películas no encontrado en', movies_path)
    if not os.path.exists(tv_path):
        print('Advertencia: Archivo de series de TV no encontrado en', tv_path)

# Ensure paths are defined, even if None for missing files, to prevent NameError
if movies_path is None:
    print("Error crítico: 'movies_path' no se pudo definir. Asegúrate de que el archivo esté subido o en la ruta correcta.")
    # Optionally, you might raise an error or exit here if paths are crucial.
if tv_path is None:
    print("Error crítico: 'tv_path' no se pudo definir. Asegúrate de que el archivo esté subido o en la ruta correcta.")
    # Optionally, you might raise an error or exit here if paths are crucial.

print('Ruta final de películas:', movies_path)
print('Ruta final de series de TV:', tv_path)


Entorno Colab detectado.
Ruta de películas (existente): /content/netflix_movies_detailed_up_to_2025.csv
Ruta de series de TV (existente): /content/netflix_tv_shows_detailed_up_to_2025.csv
Ruta final de películas: /content/netflix_movies_detailed_up_to_2025.csv
Ruta final de series de TV: /content/netflix_tv_shows_detailed_up_to_2025.csv


### Función de Limpieza y Procesamiento de Datos

Para hacer el proceso de limpieza más claro, modular y reutilizable, he encapsulado toda la lógica de limpieza en una función llamada `clean_and_process_dataframe`. Esta función realiza los siguientes pasos:

- **Carga de datos**: Lee el archivo CSV especificado.
- **Detección y Eliminación de Duplicados**: Identifica y elimina filas duplicadas, tanto filas completas como duplicados basados en `show_id` si la columna existe.
- **Limpieza de Columnas de Texto**: Elimina espacios en blanco y convierte cadenas 'nan' a `pd.NA` en columnas de tipo `object`.
- **Conversión de Fechas**: Parsea la columna `date_added` a formato de fecha y maneja errores de conversión.
- **Capitalización de País**: Convierte la columna `country` a formato de título (primera letra en mayúscula).
- **Manejo de Valores Nulos**: Reemplaza cadenas vacías, 'None' y 'nan' con `pd.NA`.
- **Cálculo de Nulos**: Cuenta los valores nulos por columna para un resumen detallado.
- **Guardar Archivo Procesado**: Guarda el DataFrame limpio en un nuevo archivo CSV en el directorio `data/processed/`.

La función también devuelve un diccionario con un resumen completo de la limpieza realizada, incluyendo el número de filas originales, duplicados encontrados, filas después de la deduplicación y el conteo de valores nulos.

In [1]:
import pandas as pd
import os
from datetime import datetime

def clean_and_process_dataframe(file_path, key, processed_dir):
    """
    Loads, cleans, and processes a single Netflix dataset (movies or TV shows).

    Args:
        file_path (str): The path to the CSV file.
        key (str): A key identifying the dataset (e.g., 'movies', 'tv').
        processed_dir (str): The directory to save the processed file.

    Returns:
        tuple: A tuple containing:
            - pd.DataFrame: The cleaned DataFrame.
            - dict: A dictionary with summary information about the cleaning process.
    """
    info = {'path': file_path}

    if not file_path or not os.path.exists(file_path):
        info['error'] = 'file not found or path is None'
        # Return empty DataFrame and info for consistency
        return pd.DataFrame(), info

    df = pd.read_csv(file_path, low_memory=False)
    info['original_rows'] = int(len(df))

    # Identify and remove duplicates
    full_dup = int(df.duplicated().sum())
    info['full_row_duplicates'] = full_dup
    if 'show_id' in df.columns:
        id_dup = int(df.duplicated(subset=['show_id']).sum())
        info['show_id_duplicates'] = id_dup
    else:
        info['show_id_duplicates'] = None
    df = df.drop_duplicates()
    info['rows_after_dedup'] = int(len(df))

    # Clean object columns: strip whitespace and convert 'nan' strings to pd.NA
    obj_cols = df.select_dtypes(include=['object']).columns.tolist()
    for c in obj_cols:
        # Ensure the column exists and is not entirely NA after initial processing
        if c in df.columns and not df[c].isnull().all():
            df[c] = df[c].astype(str).str.strip()
            df.loc[df[c] == 'nan', c] = pd.NA

    # Convert 'date_added' to datetime, coercing errors
    if 'date_added' in df.columns:
        info['date_added_parsed_nulls_before'] = int(df['date_added'].isnull().sum())
        df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
        info['date_added_parsed_nulls_after'] = int(df['date_added'].isnull().sum())

    # Capitalize 'country'
    if 'country' in df.columns:
        df['country'] = df['country'].where(df['country'].isna(), df['country'].str.title())

    # Replace specific string representations of nulls with pd.NA
    df = df.replace({'': pd.NA, 'None': pd.NA, 'nan': pd.NA})

    # Calculate null counts
    null_counts = df.isnull().sum().sort_values(ascending=False)
    info['null_counts_top10'] = null_counts.head(10).to_dict()
    info['total_nulls'] = int(null_counts.sum())

    # Save processed file
    out_path = os.path.join(processed_dir, f'{key}_processed.csv')
    df.to_csv(out_path, index=False)
    info['processed_path'] = out_path
    info['processed_rows'] = int(len(df))

    return df, info

In [10]:
import os
import pandas as pd
from datetime import datetime, timezone # Added timezone import

# Assuming movies_path and tv_path are defined in a previous cell (e.g., cell 89f49895)
# If this cell is run independently, these variables will need to be defined.

ROOT = os.getcwd()
processed_dir = os.path.join(ROOT, 'data', 'processed')
docs_dir = os.path.join(ROOT, 'documentation', 'markdown')

os.makedirs(processed_dir, exist_ok=True)
os.makedirs(docs_dir, exist_ok=True)

files_to_process = [('movies', movies_path), ('tv', tv_path)]
summary = {'run_date': datetime.now(timezone.utc).isoformat() + 'Z', 'files': {}} # Changed utcnow to now(timezone.utc)

# Iterate over the files and clean them using the new function
for key, path in files_to_process:
    df_cleaned, info = clean_and_process_dataframe(path, key, processed_dir)
    summary['files'][key] = info

print('Procesamiento completo. Archivos guardados en data/processed/')
print('Resumen de la limpieza:', summary) # Added for immediate feedback

Procesamiento completo. Archivos guardados en data/processed/
Resumen de la limpieza: {'run_date': '2026-08-28T17:41:05.772456+00:00Z', 'files': {'movies': {'path': '/content/netflix_movies_detailed_up_to_2025.csv', 'original_rows': 16000, 'full_row_duplicates': 0, 'show_id_duplicates': 0, 'rows_after_dedup': 16000, 'date_added_parsed_nulls_before': 0, 'date_added_parsed_nulls_after': 0, 'null_counts_top10': {'duration': 16000, 'country': 466, 'cast': 204, 'director': 132, 'description': 132, 'genres': 107, 'type': 0, 'show_id': 0, 'title': 0, 'date_added': 0}, 'total_nulls': 17041, 'processed_path': '/content/data/processed/movies_processed.csv', 'processed_rows': 16000}, 'tv': {'path': '/content/netflix_tv_shows_detailed_up_to_2025.csv', 'original_rows': 16000, 'full_row_duplicates': 0, 'show_id_duplicates': 9, 'rows_after_dedup': 16000, 'date_added_parsed_nulls_before': 0, 'date_added_parsed_nulls_after': 0, 'null_counts_top10': {'director': 10965, 'description': 3208, 'country': 17

In [9]:
# Escribir reporte MD con resumen de la limpieza
import os
from datetime import datetime, timezone # Added timezone import

md_path = os.path.join(docs_dir, 'EP1_data_cleaning.md')
with open(md_path, 'w', encoding='utf-8') as f:
    f.write('---\n')
    f.write("title: \"EP1 — Data cleaning inicial\"\n")
    f.write("author: \"Equipo StreamView (notebook)\"\n")
    f.write(f"date: {datetime.now(timezone.utc).date()}\n") # Corrected to use timezone.utc
    f.write('source_files:\n')
    # The 'files' variable used here refers to a tuple list, not the dict in summary
    # Re-using files_to_process from cell 2b1f1917 to list original source files
    for _, p in files_to_process:
        f.write(f"  - {p}\n")
    f.write('---\n\n')
    f.write('# Resumen de la limpieza inicial\n\n')
    f.write(f'Fecha de ejecución (UTC): {summary["run_date"]}\n\n')
    for key, info in summary['files'].items():
        f.write(f'## Archivo: {key}\n\n')
        if 'error' in info:
            f.write(f'- Error: {info["error"]}\n\n')
            continue
        f.write(f'- Ruta original: {info["path"]}\n')
        f.write(f'- Filas originales: {info.get("original_rows")}\n')
        f.write(f'- Filas después de eliminar duplicados: {info.get("rows_after_dedup")}\n')
        f.write(f'- Duplicados (filas completas): {info.get("full_row_duplicates")}\n')
        if info.get('show_id_duplicates') is not None:
            f.write(f'- Duplicados por `show_id`: {info.get("show_id_duplicates")}\n')
        if 'date_added_parsed_nulls_before' in info:
            f.write(f'- `date_added` nulos antes: {info.get("date_added_parsed_nulls_before")}\n')
            f.write(f'- `date_added` nulos después de parseo: {info.get("date_added_parsed_nulls_after")}\n')
        f.write(f'- Filas procesadas guardadas en: {info.get("processed_path")}\n')
        f.write(f'- Total de valores nulos (suma por columnas): {info.get("total_nulls")}\n')
        f.write('\n')
        f.write('### Top 10 columnas por valores faltantes\n\n')
        f.write('| Columna | Nulos |\n')
        f.write('|---|---:|\n')
        for col, n in info.get('null_counts_top10', {}).items():
            f.write(f'| {col} | {n} |\n')
        f.write('\n')
print(f'Reporte escrito en: {md_path}')

Reporte escrito en: /content/documentation/markdown/EP1_data_cleaning.md
